# Task 1 — WGAN-GP on CelebA 64×64

**Source code lives on GitHub — cloned automatically in Section 2.**

**Outputs written to** `./outputs/` (inside the cloned repo):
```
checkpoints/          model weights per epoch
samples/              periodic grids + samples_final_50.png + interpolation.png
training_curves.png   3-panel loss plot
losses.json           raw loss history
```

## 1 · Environment setup

In [ ]:
# Verify GPU is available
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Install dependencies from requirements.txt (cloned in the next section)
# Run this cell AFTER cloning, or manually install here as a fallback:
# !pip install -q torch torchvision matplotlib numpy

## 2 · Clone from GitHub & install dependencies

In [ ]:
import os, sys

# ── Change this to your GitHub repo URL ──────────────────────────────────────
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'
# ─────────────────────────────────────────────────────────────────────────────

REPO_NAME   = GITHUB_REPO.rstrip('/').split('/')[-1].replace('.git', '')
CLONE_DIR   = f'/content/{REPO_NAME}'

# Clone (or pull if already cloned — safe to re-run after a restart)
if not os.path.exists(CLONE_DIR):
    os.system(f'git clone {GITHUB_REPO} {CLONE_DIR}')
    print(f'Cloned → {CLONE_DIR}')
else:
    os.system(f'git -C {CLONE_DIR} pull')
    print(f'Pulled latest → {CLONE_DIR}')

# Install dependencies
os.system(f'pip install -q -r {CLONE_DIR}/requirements.txt')

# Add repo to Python path and set as working directory
sys.path.insert(0, CLONE_DIR)
os.chdir(CLONE_DIR)
print('Working dir:', os.getcwd())
print('Files      :', os.listdir('.'))

In [ ]:
# Optional: if the repo is private, authenticate first with a Personal Access Token
# Replace GITHUB_REPO above with:
#   https://YOUR_TOKEN@github.com/YOUR_USERNAME/YOUR_REPO_NAME.git
#
# Or use SSH (if your Colab session has an SSH key set up):
#   git@github.com:YOUR_USERNAME/YOUR_REPO_NAME.git
#
# To save outputs back to GitHub during/after training:
#   import os
#   os.system('git add outputs/ && git commit -m "training run" && git push')

## 3 · CelebA dataset setup

In [ ]:
# ── Option A: Kaggle download (recommended — avoids Google Drive quota errors) ──
# 1. Upload your kaggle.json API key once:
# from google.colab import files; files.upload()   # select kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#
# 2. Download and unzip:
# !kaggle datasets download -d jessicali9530/celeba-dataset -p ./data --unzip
#
# Expected layout after unzip:
#   ./data/celeba/img_align_celeba/*.jpg
#   ./data/celeba/list_attr_celeba.txt
#   ./data/celeba/list_eval_partition.txt

# ── Option B: torchvision auto-download (may hit GDrive quota) ──
# import torchvision
# torchvision.datasets.CelebA(root='./data', download=True)

DATA_ROOT = './data'     # <-- change if your data is elsewhere
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
# Quick loader sanity check — should print (B,3,64,64) and range ~ [-1, 1]
from dataset import get_celeba_loader

loader = get_celeba_loader(
    root=DATA_ROOT,
    split='train',
    batch_size=16,
    num_workers=2,
    max_samples=512,   # small for this check only
)
batch = next(iter(loader))
print('Batch shape :', batch.shape)
print('Value range :', batch.min().item(), '→', batch.max().item())
print('Dataset size:', len(loader.dataset), 'images')

In [ ]:
# Visualise a batch of real images
import torchvision.utils as vutils
import matplotlib.pyplot as plt

grid = vutils.make_grid(batch[:16], nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 3))
plt.axis('off')
plt.title('Sample real CelebA images (64×64)')
plt.imshow(grid.permute(1, 2, 0).cpu())
plt.tight_layout()
plt.show()

## 4 · Model sanity check

In [ ]:
import torch
from models import Generator, Critic, weights_init
from utils  import gradient_penalty

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LATENT_DIM   = 128
FEATURE_MAPS = 64

G = Generator(latent_dim=LATENT_DIM, feature_maps=FEATURE_MAPS).to(device)
D = Critic(feature_maps=FEATURE_MAPS).to(device)
G.apply(weights_init)
D.apply(weights_init)

z    = torch.randn(8, LATENT_DIM, device=device)
fake = G(z)
real = torch.randn(8, 3, 64, 64, device=device)   # dummy
gp   = gradient_penalty(D, real, fake.detach(), device)

print(f'G output : {fake.shape}  range [{fake.min():.2f}, {fake.max():.2f}]')
print(f'D output : {D(fake).shape}')
print(f'GP value : {gp.item():.4f}')
print(f'G params : {sum(p.numel() for p in G.parameters()):,}')
print(f'D params : {sum(p.numel() for p in D.parameters()):,}')

## 5 · Training configuration

In [ ]:
from train import Config

cfg = Config(
    data_root      = DATA_ROOT,
    max_samples    = None,       # None = full CelebA (~162k); set e.g. 50_000 on free tier
    latent_dim     = 128,
    feature_maps   = 64,
    num_epochs     = 20,
    batch_size     = 64,
    n_critic       = 5,
    lambda_gp      = 10.0,
    lr             = 1e-4,
    adam_b1        = 0.0,
    adam_b2        = 0.9,
    output_dir     = './outputs',
    sample_interval= 200,        # save sample grid every 200 G-steps
    ckpt_interval  = 5,          # checkpoint every 5 epochs
    num_workers    = 2,
    seed           = 42,
)
print(cfg)

## 6 · Run training

In [ ]:
from train import train

# To resume from a checkpoint:
#   train(cfg, resume_from='./outputs/checkpoints/ckpt_epoch005.pt')

train(cfg, resume_from=None)

## 7 · Training curves

In [ ]:
from utils import LossTracker, plot_training_curves
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

# Reload from JSON in case the training cell was re-run
tracker = LossTracker.load('./outputs/losses.json')
print(f'Total G steps recorded: {len(tracker)}')

# Re-render the 3-panel figure inline
plot_training_curves(
    tracker,
    save_path='./outputs/training_curves.png',
    show=False,      # we display below
    ema_alpha=0.05,
)
display(IPImage('./outputs/training_curves.png'))

## 8 · 50 generated samples (report grid)

In [ ]:
import torch, matplotlib.pyplot as plt
import torchvision.utils as vutils
from models import Generator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the final checkpoint
import glob, os
ckpts = sorted(glob.glob('./outputs/checkpoints/ckpt_epoch*.pt'))
print('Available checkpoints:', [os.path.basename(c) for c in ckpts])

G = Generator(latent_dim=cfg.latent_dim, feature_maps=cfg.feature_maps).to(device)
ckpt = torch.load(ckpts[-1], map_location=device)   # latest checkpoint
G.load_state_dict(ckpt['G'])
G.eval()
print(f'Loaded: {os.path.basename(ckpts[-1])}')

with torch.no_grad():
    z50   = torch.randn(50, cfg.latent_dim, device=device)
    imgs  = G(z50)

grid = vutils.make_grid(imgs, nrow=10, normalize=True, value_range=(-1, 1), padding=2)

plt.figure(figsize=(18, 9))
plt.axis('off')
plt.title('50 Generated CelebA Samples — WGAN-GP', fontsize=14)
plt.imshow(grid.permute(1, 2, 0).cpu())
plt.tight_layout()
plt.savefig('./outputs/samples/samples_final_50.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Latent interpolation (≥ 8 intermediate steps)

In [ ]:
import torch, matplotlib.pyplot as plt
import torchvision.utils as vutils
from train import latent_interpolation

# G should still be loaded from the previous cell
latent_interpolation(
    G,
    device=device,
    latent_dim=cfg.latent_dim,
    n_steps=10,              # 10 intermediate + 2 endpoints = 12 frames
    save_path='./outputs/samples/interpolation.png',
    seed=cfg.seed,
)

interp = plt.imread('./outputs/samples/interpolation.png')
plt.figure(figsize=(20, 3))
plt.axis('off')
plt.title('Latent Interpolation  z₁ → z₂  (10 intermediate steps)', fontsize=13)
plt.imshow(interp)
plt.tight_layout()
plt.show()

## 10 · Browse periodic sample grids
Useful for spotting when the generator started producing coherent faces.

In [ ]:
import glob, os
import matplotlib.pyplot as plt

grids = sorted(glob.glob('./outputs/samples/grid_*.png'))
print(f'Found {len(grids)} periodic grids')

# Show every 5th grid to keep the notebook compact
show_grids = grids[::5] if len(grids) > 5 else grids

fig, axes = plt.subplots(1, len(show_grids), figsize=(5 * len(show_grids), 5))
if len(show_grids) == 1:
    axes = [axes]
for ax, path in zip(axes, show_grids):
    step = os.path.basename(path).replace('grid_', '').replace('.png', '')
    ax.imshow(plt.imread(path))
    ax.set_title(f'step {int(step):,}', fontsize=9)
    ax.axis('off')
plt.suptitle('Generator progress over training', fontsize=12)
plt.tight_layout()
plt.show()

## 11 · Download all outputs as a zip

In [ ]:
# Zip the outputs folder and download it locally
# Skip if you are saving directly to Drive.
import shutil
shutil.make_archive('wgan_gp_outputs', 'zip', './outputs')

from google.colab import files
files.download('wgan_gp_outputs.zip')